In [1]:
import sys; sys.path.append('..')

In [2]:
from malign_logits import *

In [3]:
base_model, instruct_model, tokenizer = load_models()

Detected Mac - using MPS (Metal Performance Shaders) with torch.float16
Loading base model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading instruct model...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both models loaded.


In [4]:
# !pip install -r ../requirements.txt

In [ ]:
# prompt = "She began to suck his "
prompt = "She was so angry she wanted to "

ego_words = discover_top_words(instruct_model, tokenizer, prompt)
print("Ego top 10:", list(ego_words.keys())[:10])

100%|██████████| 200/200 [00:38<00:00,  5.16it/s]

Ego top 10: ['cock', 'tongue', 'erect', 'hard', 'ube', 'penis', 'iced', 'ropy', 'mouth', 'finger']


In [17]:
base_words = discover_top_words(base_model, tokenizer, prompt)
print("Base top 10:", list(base_words.keys())[:10])

100%|██████████| 200/200 [00:42<00:00,  4.65it/s]

Base top 10: ['ropy', 'cock', 'ingle', 'inger', 'hard', 'icks', 'ʙᴜ', 'ick', 'umber', 'ube']


In [18]:
superego_prompt = make_superego_prompt(prompt)
superego_words = discover_top_words(instruct_model, tokenizer, superego_prompt)

100%|██████████| 200/200 [00:37<00:00,  5.38it/s]


In [19]:
df = compute_repression(ego_words, superego_words)

In [22]:
df

,word,ego,superego,delta,ratio,repressed,amplified
8,cock,0.388615,0.009113,0.379502,4.264482e+01,True,False
112,tongue,0.084423,0.013068,0.071356,6.460546e+00,True,False
16,hard,0.065271,0.023021,0.042250,2.835312e+00,True,False
145,ube,0.037626,0.000000,0.037626,3.762650e+08,True,False
46,penis,0.037485,0.007603,0.029882,4.930491e+00,True,False
...,...,...,...,...,...,...,...
96,between,0.000000,0.048339,-0.048339,-4.833910e+08,False,True
83,as,0.000000,0.055237,-0.055237,-5.523671e+08,False,True
37,on,0.000000,0.072811,-0.072811,-7.281081e+08,False,True
94,in,0.004667,0.082492,-0.077825,-1.767612e+01,False,True


In [31]:
import torch

def compute_displacement_v2(
    base_words,
    ego_words,
    superego_words,
    base_logits,
    embeddings,
    tokenizer,
    similarity_threshold=0.3,    # Much higher — only genuinely similar words
    min_repression=0.005,         # Ignore trivially repressed words
    min_displacement_mass=0.0005, # Don't log tiny displacements
):
    """
    Displacement engine v2 — fixes self-displacement and noise issues.
    """
    from collections import defaultdict

    all_words = set(base_words.keys()) | set(ego_words.keys()) | set(superego_words.keys())

    # Identify genuinely repressed words (ego wants, superego suppresses)
    repressed = {}
    for w in all_words:
        e = ego_words.get(w, 0)
        s = superego_words.get(w, 0)
        repression = e - s
        if repression > min_repression:
            repressed[w] = {
                'mass': repression,
                'ego': e,
                'superego': s,
                'base': base_words.get(w, 0),
            }

    # Identify permitted words (NOT repressed) as displacement targets
    permitted = {}
    for w, prob in superego_words.items():
        if w not in repressed and prob > 0.001:
            permitted[w] = prob

    if not repressed or not permitted:
        return dict(superego_words), {}

    # Precompute embeddings for repressed and permitted words
    base_probs = torch.softmax(base_logits.float(), dim=-1)
    normed_emb = torch.nn.functional.normalize(embeddings.float(), dim=-1)

    def get_emb(word):
        tokens = tokenizer.encode(word, add_special_tokens=False)
        if tokens:
            return normed_emb[tokens[0]]
        return None

    repressed_embs = {w: get_emb(w) for w in repressed}
    permitted_embs = {w: get_emb(w) for w in permitted}

    # Remove words we can't embed
    repressed_embs = {w: e for w, e in repressed_embs.items() if e is not None}
    permitted_embs = {w: e for w, e in permitted_embs.items() if e is not None}

    # Compute similarity matrix: repressed × permitted
    displaced = dict(superego_words)
    condensation_log = defaultdict(list)

    for rep_word, rep_info in repressed.items():
        if rep_word not in repressed_embs:
            continue

        rep_emb = repressed_embs[rep_word]
        mass_to_displace = rep_info['mass']

        # Score each permitted word
        targets = {}
        for perm_word, perm_emb in permitted_embs.items():
            sim = torch.dot(rep_emb, perm_emb).item()

            if sim < similarity_threshold:
                continue

            # Weight by base model drive energy
            perm_tokens = tokenizer.encode(perm_word, add_special_tokens=False)
            drive = base_probs[perm_tokens[0]].item() if perm_tokens else 0
            targets[perm_word] = sim * (1 + drive * 10)

        if not targets:
            continue

        # Distribute mass
        total_weight = sum(targets.values())
        for target_word, weight in targets.items():
            added = mass_to_displace * (weight / total_weight)

            if added < min_displacement_mass:
                continue

            displaced[target_word] = displaced.get(target_word, 0) + added
            condensation_log[target_word].append({
                'source': rep_word,
                'mass': added,
                'similarity': torch.dot(
                    repressed_embs[rep_word], 
                    permitted_embs[target_word]
                ).item(),
            })

    # Normalize
    total = sum(displaced.values())
    if total > 0:
        displaced = {w: p / total for w, p in displaced.items()}

    displaced = dict(sorted(displaced.items(), key=lambda x: -x[1]))
    return displaced, dict(condensation_log)

In [32]:
def compute_displacement_v3(
    base_words,
    ego_words,
    superego_words,
    model,
    tokenizer,
    prompt,
    similarity_threshold=0.3,
    min_repression=0.005,
    min_displacement_mass=0.0005,
    hidden_layer=16,
    device=None,
):
    """
    Displacement engine v3 — uses contextual embeddings from a 
    hidden layer instead of input embeddings for semantic similarity.
    Also filters out morphological variants (same stem).
    """
    from collections import defaultdict

    if device is None:
        device = next(model.parameters()).device

    all_words = set(base_words.keys()) | set(ego_words.keys()) | set(superego_words.keys())

    # Identify repressed and permitted words
    repressed = {}
    for w in all_words:
        e = ego_words.get(w, 0)
        s = superego_words.get(w, 0)
        if e - s > min_repression:
            repressed[w] = {'mass': e - s, 'ego': e, 'superego': s}

    permitted = {}
    for w, prob in superego_words.items():
        if w not in repressed and prob > 0.001:
            permitted[w] = prob

    if not repressed or not permitted:
        return dict(superego_words), {}

    # Get contextual embeddings by running each word through the model
    # in the context of the prompt
    def get_contextual_embedding(word):
        """Get the hidden state for a word when it follows the prompt."""
        text = prompt + " " + word
        ids = tokenizer.encode(text, return_tensors="pt").to(device)
        prompt_len = len(tokenizer.encode(prompt))

        with torch.no_grad():
            outputs = model(ids, output_hidden_states=True)
            # Get the specified hidden layer's output
            hidden = outputs.hidden_states[hidden_layer]
            # Average the hidden states for all tokens belonging to the word
            word_hidden = hidden[0, prompt_len:, :].mean(dim=0).cpu()

        return word_hidden

    print("  Computing contextual embeddings for repressed words...")
    rep_embs = {}
    for w in repressed:
        try:
            rep_embs[w] = get_contextual_embedding(w)
        except Exception:
            continue

    print("  Computing contextual embeddings for permitted words...")
    perm_embs = {}
    for w in permitted:
        try:
            perm_embs[w] = get_contextual_embedding(w)
        except Exception:
            continue

    # Normalize
    for w in rep_embs:
        rep_embs[w] = torch.nn.functional.normalize(rep_embs[w].unsqueeze(0), dim=-1).squeeze()
    for w in perm_embs:
        perm_embs[w] = torch.nn.functional.normalize(perm_embs[w].unsqueeze(0), dim=-1).squeeze()

    def shares_stem(w1, w2, min_shared=4):
        """Filter out morphological variants."""
        w1, w2 = w1.lower(), w2.lower()
        if w1 in w2 or w2 in w1:
            return True
        # Check shared prefix
        shared = 0
        for a, b in zip(w1, w2):
            if a == b:
                shared += 1
            else:
                break
        return shared >= min_shared and shared >= 0.6 * min(len(w1), len(w2))

    # Displacement
    displaced = dict(superego_words)
    condensation_log = defaultdict(list)

    for rep_word, rep_info in repressed.items():
        if rep_word not in rep_embs:
            continue

        mass = rep_info['mass']
        targets = {}

        for perm_word, perm_emb in perm_embs.items():
            # Skip morphological variants
            if shares_stem(rep_word, perm_word):
                continue

            sim = torch.dot(rep_embs[rep_word], perm_emb).item()
            if sim < similarity_threshold:
                continue

            targets[perm_word] = sim

        if not targets:
            continue

        total_weight = sum(targets.values())
        for target, weight in targets.items():
            added = mass * (weight / total_weight)
            if added < min_displacement_mass:
                continue

            displaced[target] = displaced.get(target, 0) + added
            condensation_log[target].append({
                'source': rep_word,
                'mass': added,
                'similarity': weight,
            })

    # Normalize
    total = sum(displaced.values())
    if total > 0:
        displaced = {w: p / total for w, p in displaced.items()}

    displaced = dict(sorted(displaced.items(), key=lambda x: -x[1]))
    return displaced, dict(condensation_log)

In [35]:
# Step 2: Get base model logits and embeddings (needed for semantic similarity)
base_logits = get_base_logits(base_model, tokenizer, prompt)
embeddings = get_embeddings(base_model)

# # Step 3: Run displacement
# displaced_dist, condensation_log = compute_displacement_v3(
#     base_words, ego_words, superego_words,
#     base_logits, embeddings, tokenizer,
# )

displaced_dist, condensation_log = compute_displacement_v3(
    base_words, ego_words, superego_words,
    instruct_model,       # the model, not logits
    tokenizer,
    prompt,               # the raw prompt string
)

  Computing contextual embeddings for repressed words...
  Computing contextual embeddings for permitted words...


In [36]:

# Step 4: Examine results

# The neurotic distribution — superego's vocabulary but with displaced charge
print("=== NEUROTIC DISTRIBUTION ===\n")
for word, prob in list(displaced_dist.items())[:20]:
    sup = superego_words.get(word, 0)
    ego = ego_words.get(word, 0)
    gained = prob - sup
    marker = ""
    if gained > 0.01:
        marker = " ← SYMPTOMATIC"
    elif gained > 0.005:
        marker = " ← trace"
    print(f"{word:20s}  neurotic: {prob:.4f}  superego: {sup:.4f}  "
          f"ego: {ego:.4f}  displaced_mass: {gained:+.4f}{marker}")

# Condensation points — words absorbing charge from multiple repressed sources
print("\n=== CONDENSATION POINTS ===\n")
for word, sources in sorted(condensation_log.items(), 
                             key=lambda x: sum(s['mass'] for s in x[1]), 
                             reverse=True)[:10]:
    source_words = [s["source"] for s in sources]
    total_mass = sum(s["mass"] for s in sources)
    n_sources = len(sources)
    print(f"'{word}' absorbed {total_mass:.4f} from {n_sources} repressed words:")
    for s in sources:
        print(f"    ← '{s['source']}' ({s['mass']:.4f})")

=== NEUROTIC DISTRIBUTION ===

and                   neurotic: 0.0906  superego: 0.1305  ego: 0.0134  displaced_mass: -0.0399
in                    neurotic: 0.0598  superego: 0.0825  ego: 0.0047  displaced_mass: -0.0227
erect                 neurotic: 0.0556  superego: 0.0755  ego: 0.0779  displaced_mass: -0.0199
on                    neurotic: 0.0526  superego: 0.0728  ego: 0.0000  displaced_mass: -0.0202
as                    neurotic: 0.0421  superego: 0.0552  ego: 0.0000  displaced_mass: -0.0131
between               neurotic: 0.0367  superego: 0.0483  ego: 0.0000  displaced_mass: -0.0116
when                  neurotic: 0.0362  superego: 0.0465  ego: 0.0000  displaced_mass: -0.0103
into                  neurotic: 0.0345  superego: 0.0433  ego: 0.0000  displaced_mass: -0.0088
insert                neurotic: 0.0256  superego: 0.0328  ego: 0.0000  displaced_mass: -0.0071
because               neurotic: 0.0223  superego: 0.0246  ego: 0.0000  displaced_mass: -0.0023
to                 